# Smoke tests da divisão pelo CFL no StoCFL

Gráficos dos smoke tests da [hipótese da divisão](../../docs/hipotese-stocfl-divisao-cfl.md) (seções 13 e 14; D-058 a D-061), rodados com o [StoCFL-FCC+CFL-split.py](StoCFL-FCC+CFL-split.py): `n = 40`, `k = 4` grupos de 10 clientes, todos os clientes em toda rodada, 40 rodadas, seeds 0 e 1, sem dropout.

- **`PARTICAO`** escolhe o conjunto de dados:
  - `"classes"` (**padrão desde a D-061**): MNIST sem rotação, cada grupo com só as classes {0,1,2}, {3,4}, {5,6} ou {7,8,9} (`--label_groups 1`), com cenários tudo fundido (`τ = −0.3`), um par encadeado ({3,4} com {5,6}, `τ = −0.22`) e controle (`τ = 0.5`);
  - `"rotacao"`: Rotated MNIST (0°, 90°, 180°, 270°), com tudo fundido (`τ = 0.25`), um par encadeado (`τ` = 0.28 / 0.26) e controle (`τ = 0.6`).
- **Variantes:** `nosplit` (StoCFL sem divisão), `none` (divisão sem memória), `reanchor` (novo modelo global: o `Ψ` de todos é recalculado no global treinado) e `cannot_link` (as partes não podem voltar a se juntar). No controle só `nosplit` e `none`.
- **Política 7:** com `DADOS_JSON = True` (padrão) o notebook só lê `logs/resultados-divisao-cfl-smoke.json.gz` e plota, sem rodar nada. Com `DADOS_JSON = False` ele roda as execuções que faltarem (~10 min cada, 3 em paralelo na GPU), grava o JSON e apaga as pastas de logs. Sem os logs e com o JSON já gravado, ele para com um aviso, a menos que `REEXECUTAR = True`.
- Use o kernel `.venv` da raiz. O notebook roda aberto da raiz do repositório ou de `projetos/StoCFL/` (ele muda para `projetos/StoCFL/` sozinho).

In [ ]:
PARTICAO = "classes"   # "classes" (MNIST sem rotação, grupos de classes; padrão) ou "rotacao"
DADOS_JSON = True      # True: só lê o JSON e plota
REEXECUTAR = False     # True: roda a grade de novo mesmo com o JSON gravado
PARALELO = 3           # execuções simultâneas na GPU
APAGAR_LOGS = True     # apaga as pastas de logs depois de gravar e conferir o JSON

EPS_CLASSES = (0.07, 0.378)   # calibrados na D-061 nas execuções sem divisão
CONFIG = {
    "rotacao": {"json": "resultados-divisao-cfl-smoke.json.gz", "args": [], "tag": "", "eps": (0.09, 0.146),   # D-058
                "grupos": ["0°", "90°", "180°", "270°"],
                "cenarios": {"tudo-junto": {0: 0.25, 1: 0.25}, "um-encadeamento": {0: 0.28, 1: 0.26}, "controle": {0: 0.6, 1: 0.6}}},
    "classes": {"json": "resultados-divisao-cfl-classes.json.gz", "args": ["--label_groups", "1"], "tag": "-lg", "eps": EPS_CLASSES,  # D-061
                "grupos": ["{0,1,2}", "{3,4}", "{5,6}", "{7,8,9}"],
                "cenarios": {"tudo-junto": {0: -0.3, 1: -0.3}, "um-encadeamento": {0: -0.22, 1: -0.22}, "controle": {0: 0.5, 1: 0.5}}},
}[PARTICAO]
ROUNDS, GAMMA_MAX = 40, 0.5
EPS1, EPS2 = CONFIG["eps"]
SEEDS = [0, 1]
CENARIOS = CONFIG["cenarios"]
VARIANTES = {"tudo-junto": ["nosplit", "none", "reanchor", "cannot_link"],
             "um-encadeamento": ["nosplit", "none", "reanchor", "cannot_link"],
             "controle": ["nosplit", "none"]}
N, K, JANELA = 40, 4, 10

In [ ]:
import gzip, json, os, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# funciona aberto da raiz do repositório ou de projetos/StoCFL/: o resto do notebook usa caminhos relativos a projetos/StoCFL/
for _base in (Path("."), Path("projetos/StoCFL"), Path("../projetos/StoCFL")):
    if (_base / "StoCFL-FCC+CFL-split.py").exists():
        os.chdir(_base)
        break
else:
    raise RuntimeError("rode o notebook da raiz do repositório ou de projetos/StoCFL/")
print("pasta de trabalho:", Path.cwd())
LOGS = Path("logs")
SAIDA = LOGS / CONFIG["json"]
GRUPO = np.arange(N) * K // N
NOMES = np.array(CONFIG["grupos"])[GRUPO]   # nome do grupo de cada cliente


def pasta_de(cenario, seed, variante):
    tau = CENARIOS[cenario][seed]
    tag = "nosplit" if variante == "nosplit" else "split-" + variante
    achadas = sorted(LOGS.glob(f"runs-split-rotated-mnist-n{N}-p{N}-seed{seed}-lambda0.05{CONFIG['tag']}-tau{tau}-{tag}-time-*"))
    return achadas[-1] if achadas else None


def rodar(cenario, seed, variante):
    args = [sys.executable, "StoCFL-FCC+CFL-split.py", "--n", str(N), "--k", str(K), "--dataset", "mnist",
            "--com_round", str(ROUNDS), "--num_per_round", str(N), "--train", "1",
            "--tau", str(CENARIOS[cenario][seed]), "--seed", str(seed), "--process", "0"] + CONFIG["args"]
    if variante == "nosplit":
        args += ["--split", "0"]
    else:
        args += ["--split", "1", "--after_split", variante, "--eps1", str(EPS1), "--eps2", str(EPS2), "--gamma_max", str(GAMMA_MAX)]
    t0 = time.time()
    with open(LOGS / f"saida-divisao-{cenario}-s{seed}-{variante}.txt", "w") as f:
        rc = subprocess.run(args, stdout=f, stderr=subprocess.STDOUT).returncode
    assert rc == 0, (cenario, seed, variante)
    return time.time() - t0


def carregar(cenario, seed, variante, duracao=None):
    import torch
    p = pasta_de(cenario, seed, variante)
    L = lambda f: torch.load(p / f, weights_only=False)
    rot = []
    for part in L("cluster.pkl"):
        r = np.empty(N, dtype=int)
        for c, mem in part.items():
            r[mem] = c
        rot.append(r.tolist())
    acc, ev = L("accuracy_clients.pkl"), L("split_events.pkl")
    return {"cenario": cenario, "seed": seed, "variante": variante, "tau": CENARIOS[cenario][seed], "rounds": ROUNDS,
            "eps1": None if variante == "nosplit" else EPS1, "eps2": None if variante == "nosplit" else EPS2,
            "gamma_max": None if variante == "nosplit" else GAMMA_MAX, "pasta": p.name, "duracao_s": duracao,
            "rotulos_por_rodada": rot,
            "acc_cluster_por_cliente": [[None if a is None else float(a) for a in r["cluster"]] for r in acc],
            "acc_global_por_grupo": [r["global"] for r in acc],
            "normas_por_rodada": [[{**e, "members": list(map(int, e["members"]))} for e in rnd] for rnd in L("norms.pkl")],
            "divisoes": [{**d, "sides": [list(map(int, s)) for s in d["sides"]]} for d in ev["splits"]],
            "refusoes": ev["remerges"], "bloqueadas": ev["blocked"], "reancoragens": ev["reanchors"],
            "psi_cos": [{"rodada": e["round"], "motivo": e["why"], "cos": np.round(e["cos"].astype(float), 6).tolist()} for e in L("psi_cos.pkl")],
            "linha_args": (p / "rotated_mnist.log").read_text().splitlines()[0]}


def gravar(dados):
    tmp = SAIDA.with_suffix(".tmp")
    with open(tmp, "wb") as f, gzip.GzipFile(fileobj=f, mode="wb", compresslevel=9, mtime=0) as gz:
        gz.write(json.dumps(dados, allow_nan=False).encode())
    os.replace(tmp, SAIDA)


combos = [(c, s, v) for c in CENARIOS for s in SEEDS for v in VARIANTES[c]]
if DADOS_JSON:
    DADOS = json.loads(gzip.open(SAIDA).read())
else:
    faltam = [x for x in combos if REEXECUTAR or pasta_de(*x) is None]
    if faltam and SAIDA.exists() and not REEXECUTAR:
        raise RuntimeError("Os logs foram apagados e o JSON já tem os resultados: use DADOS_JSON = True "
                           "(ou REEXECUTAR = True para rodar a grade de novo, ~1 h na GPU).")
    with ThreadPoolExecutor(PARALELO) as ex:
        duracoes = dict(zip(faltam, ex.map(lambda x: rodar(*x), faltam)))
    anterior = json.loads(gzip.open(SAIDA).read()) if SAIDA.exists() else {}
    DADOS = {k: v for k, v in anterior.items() if k != "execucoes"}
    DADOS.update({"descricao": "smoke tests da divisao pelo CFL (docs/hipotese-stocfl-divisao-cfl.md, secao 13)",
                  "parametros": {"n": N, "k": K, "m": N, "dataset": "mnist (rotacao)", "janela_final": JANELA}})
    DADOS["execucoes"] = [carregar(*x, duracoes.get(x)) for x in combos]
    gravar(DADOS)
    if APAGAR_LOGS:
        conf = json.loads(gzip.open(SAIDA).read())
        assert conf["execucoes"] == json.loads(json.dumps(DADOS["execucoes"]))
        for x in combos:
            shutil.rmtree(pasta_de(*x))
        for f in LOGS.glob("saida-divisao-*.txt"):
            f.unlink()
E = {(e["cenario"], e["seed"], e["variante"]): e for e in DADOS["execucoes"]}
print(len(E), "execuções")

## Resumo

Acurácia = a do modelo do cluster de cada cliente no teste do grupo dele, média dos 40 clientes, na última rodada e na média das últimas `JANELA` rodadas. "Voltas" são fusões que juntam de novo clientes separados por uma divisão.

In [ ]:
def resumo(e):
    from sklearn.metrics import adjusted_rand_score
    rot = np.array(e["rotulos_por_rodada"]); acc = np.array(e["acc_cluster_por_cliente"], dtype=float)
    aceitas = [d for d in e["divisoes"] if d["accepted"]]
    return {"cenário": e["cenario"], "seed": e["seed"], "variante": e["variante"], "τ": e["tau"],
            "clusters no fim": len(set(rot[-1])), "ARI": round(adjusted_rand_score(GRUPO, rot[-1]), 3),
            "divisões": len(aceitas), "1ª divisão": aceitas[0]["round"] if aceitas else None,
            "voltas": len(e["refusoes"]), "bloqueadas": len(e["bloqueadas"]), "reancoragens": len(e["reancoragens"]),
            "acc última": round(np.nanmean(acc[-1]), 3), "acc janela": round(np.nanmean(acc[-JANELA:]), 3),
            "acc global": round(float(np.mean(e["acc_global_por_grupo"][-1])), 3)}
TABELA = pd.DataFrame([resumo(e) for e in DADOS["execucoes"]]).sort_values(["cenário", "seed", "variante"])
TABELA

## Evolução dos clusters

Cada painel é uma execução: no eixo horizontal as rodadas, no vertical os 40 clientes (agrupados por rotação), e a cor diz **o que o cluster do cliente contém** naquela rodada (por exemplo, "{3,4}+{5,6}" ou "0°+180°" é um cluster com clientes de dois grupos; uma cor de um grupo só é um cluster puro). Os clientes começam sozinhos e o StoCFL os agrupa (rodadas 0 e 1); as linhas verticais verdes marcam as **divisões do CFL** e as vermelhas as **voltas** (as partes fundidas de novo pelo StoCFL). Passe o mouse para ver o id do cluster.

In [ ]:
def composicao(rot):
    # nome de cada cluster pela rotação dos membros, por rodada
    out = []
    for r in rot:
        r = np.array(r); nomes = {}
        for c in np.unique(r):
            nomes[c] = "+".join(n for n in CONFIG["grupos"] if n in set(NOMES[r == c]))
        out.append([nomes[c] for c in r])
    return np.array(out).T   # clientes x rodadas

TODOS = sorted({n for e in DADOS["execucoes"] for n in composicao(e["rotulos_por_rodada"]).ravel()},
               key=lambda s: (s.count("+"), s))
PALETA = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2", "#EECA3B", "#9D755D", "#FF9DA6", "#BAB0AC", "#2F2F2F"]
COR = {n: PALETA[i % len(PALETA)] for i, n in enumerate(TODOS)}


def evolucao(cenario, seed):
    vs = VARIANTES[cenario]
    fig = make_subplots(rows=len(vs), cols=1, shared_xaxes=True, vertical_spacing=0.04,
                        subplot_titles=[f"{v}" for v in vs])
    for i, v in enumerate(vs, 1):
        e = E[(cenario, seed, v)]
        comp = composicao(e["rotulos_por_rodada"])
        ids = np.array(e["rotulos_por_rodada"]).T
        z = np.vectorize(TODOS.index)(comp)
        escala = []
        for j, n in enumerate(TODOS):
            a, b = j / len(TODOS), (j + 1) / len(TODOS)
            escala += [[a, COR[n]], [b, COR[n]]]
        fig.add_trace(go.Heatmap(z=z, zmin=-0.5, zmax=len(TODOS) - 0.5, colorscale=escala, showscale=False,
                                 customdata=np.dstack([comp, ids]), y=[f"{NOMES[c]} · cliente {c}" for c in range(N)],
                                 hovertemplate="rodada %{x}<br>%{y}<br>cluster %{customdata[1]}: %{customdata[0]}<extra></extra>"),
                      row=i, col=1)
        for d in e["divisoes"]:
            if d["accepted"]:
                fig.add_vline(x=d["round"] + 0.5, line_color="green", line_width=2, row=i, col=1)
        for r in {x["round"] for x in e["refusoes"]}:
            fig.add_vline(x=r - 0.5, line_color="red", line_width=1, line_dash="dot", row=i, col=1)
        fig.update_yaxes(showticklabels=False, row=i, col=1, title_text="clientes")
    for n in TODOS:   # legenda das cores
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(color=COR[n], size=12, symbol="square"),
                                 name=n, showlegend=True))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="green"), name="divisão do CFL"))
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="red", dash="dot"), name="volta (refusão)"))
    fig.update_xaxes(title_text="rodada", row=len(vs), col=1)
    fig.update_layout(height=260 * len(vs) + 80, title=f"Evolução dos clusters: {cenario}, seed {seed} (τ = {CENARIOS[cenario][seed]})",
                      legend_title="conteúdo do cluster")
    return fig


for c in CENARIOS:
    for s in SEEDS:
        evolucao(c, s).show()

## Número de clusters por rodada

In [ ]:
for c in CENARIOS:
    fig = make_subplots(rows=1, cols=len(SEEDS), subplot_titles=[f"seed {s}" for s in SEEDS], shared_yaxes=True)
    for j, s in enumerate(SEEDS, 1):
        for v in VARIANTES[c]:
            n = [len(set(r)) for r in E[(c, s, v)]["rotulos_por_rodada"]]
            fig.add_trace(go.Scatter(y=n, mode="lines", name=v, legendgroup=v, showlegend=j == 1), row=1, col=j)
    fig.update_yaxes(type="log", title_text="clusters", col=1)
    fig.update_xaxes(title_text="rodada")
    fig.update_layout(title=f"Clusters por rodada: {c}", height=350)
    fig.show()

## Acurácia por rodada

Média, nos 40 clientes, da acurácia do modelo do cluster de cada um no teste do grupo dele; a linha tracejada cinza é o modelo global.

In [ ]:
for c in CENARIOS:
    fig = make_subplots(rows=1, cols=len(SEEDS), subplot_titles=[f"seed {s}" for s in SEEDS], shared_yaxes=True)
    for j, s in enumerate(SEEDS, 1):
        for v in VARIANTES[c]:
            a = np.nanmean(np.array(E[(c, s, v)]["acc_cluster_por_cliente"], dtype=float), axis=1)
            fig.add_trace(go.Scatter(y=a, mode="lines", name=v, legendgroup=v, showlegend=j == 1), row=1, col=j)
        g = np.mean(E[(c, s, "nosplit")]["acc_global_por_grupo"], axis=1)
        fig.add_trace(go.Scatter(y=g, mode="lines", name="global", line=dict(color="gray", dash="dash"),
                                 legendgroup="global", showlegend=j == 1), row=1, col=j)
    fig.update_yaxes(title_text="acurácia", col=1)
    fig.update_xaxes(title_text="rodada")
    fig.update_layout(title=f"Acurácia por rodada: {c}", height=380)
    fig.show()

## Critério do CFL nas execuções sem divisão

Normas das atualizações (`ΔW`) dos membros no modelo do cluster: média (`‖média ΔW‖`, linha cheia) e máxima (`max ‖ΔW_i‖`, tracejada), para o maior cluster encadeado (com algum par de `Ψ` ≤ `τ`) e para um cluster puro. O CFL divide quando a média fica abaixo de `ε1` e a máxima acima de `ε2` (linhas horizontais). Nos encadeados a máxima estaciona alta e a média cai; nos puros as duas caem juntas.

In [ ]:
fig = make_subplots(rows=1, cols=len(SEEDS), subplot_titles=[f"seed {s}" for s in SEEDS], shared_yaxes=True)
for j, s in enumerate(SEEDS, 1):
    for c in CENARIOS:
        e = E[(c, s, "nosplit")]
        for tipo, cond in [("encadeado", lambda x: x["psi_min"] <= e["tau"]), ("puro", lambda x: x["psi_min"] > e["tau"])]:
            serie = [max([x for x in rnd if cond(x)], key=lambda x: len(x["members"]), default=None) for rnd in e["normas_por_rodada"]]
            if not any(serie) or (tipo == "puro" and c != "controle"):
                continue
            nome = f"{c}, {tipo}"
            fig.add_trace(go.Scatter(y=[x and x["mean_norm"] for x in serie], name=nome + " (média)", legendgroup=nome,
                                     showlegend=j == 1), row=1, col=j)
            fig.add_trace(go.Scatter(y=[x and x["max_norm"] for x in serie], name=nome + " (máxima)", line=dict(dash="dash"),
                                     legendgroup=nome, showlegend=j == 1), row=1, col=j)
    fig.add_hline(y=EPS1, line_color="black", annotation_text="ε1", row=1, col=j)
    fig.add_hline(y=EPS2, line_color="black", line_dash="dot", annotation_text="ε2", row=1, col=j)
fig.update_xaxes(title_text="rodada"); fig.update_yaxes(title_text="norma", col=1)
fig.update_layout(title="Critério de norma do CFL (sem divisão)", height=420)
fig.show()

## O novo modelo global: cossenos entre os `Ψ` antes e depois da reancoragem

Cosseno entre os `Ψ` dos 40 clientes (ordenados por grupo) na âncora inicial (`ω₀`, aleatória) e depois de cada reancoragem no modelo global treinado. Na âncora inicial, compare os cossenos entre grupos diferentes nas duas âncoras: é o sinal deles, contra `τ`, que decide se as partes voltam a se fundir.

In [ ]:
for c in ["tudo-junto", "um-encadeamento"]:
    for s in SEEDS:
        mats = E[(c, s, "reanchor")]["psi_cos"]
        fig = make_subplots(rows=1, cols=len(mats), subplot_titles=[f"rodada {m['rodada']} ({'âncora inicial' if m['motivo'] == 'observe' else 'reancoragem'})" for m in mats])
        rotulos = [f"{NOMES[i]} · {i}" for i in range(N)]
        for j, m in enumerate(mats, 1):
            fig.add_trace(go.Heatmap(z=m["cos"], x=rotulos, y=rotulos, zmin=-1, zmax=1, colorscale="RdBu", reversescale=True,
                                     showscale=j == len(mats), hovertemplate="%{y} × %{x}: %{z:.3f}<extra></extra>"), row=1, col=j)
            fig.update_xaxes(showticklabels=False, row=1, col=j); fig.update_yaxes(showticklabels=False, autorange="reversed", row=1, col=j)
        fig.update_layout(title=f"Cosseno entre os Ψ: {c}, seed {s}", height=380, width=330 * len(mats) + 120)
        fig.show()

## Divisões do CFL

Cada divisão aceita: rodada, cluster dividido, os dois lados (em grupos), `α_cross^max` (maior cosseno entre os `ΔW` dos dois lados; a divisão é aceita se `γ_max < √((1 − α)/2)`) e a rodada em que as partes voltaram a se juntar, se voltaram.

In [ ]:
linhas = []
for e in DADOS["execucoes"]:
    rot = np.array(e["rotulos_por_rodada"])
    for d in e["divisoes"]:
        A, B = d["sides"]
        volta = next((r for r in range(d["round"] + 1, len(rot)) if np.intersect1d(rot[r][A], rot[r][B]).size), None)
        lado = lambda S: "+".join(n for n in CONFIG["grupos"] if n in set(NOMES[S]))
        l1, l2 = sorted([lado(A), lado(B)])
        linhas.append({"cenário": e["cenario"], "seed": e["seed"], "variante": e["variante"], "rodada": d["round"],
                       "lado 1": l1, "lado 2": l2, "α_cross^max": round(d["alpha_cross_max"], 3),
                       "aceita": d["accepted"], "voltou na rodada": volta})
DIVISOES = pd.DataFrame(linhas)
# no "tudo junto" sem memória a mesma divisão se repete em toda rodada: mostra só a primeira e a contagem
DIVISOES.groupby(["cenário", "seed", "variante", "lado 1", "lado 2"], sort=False).agg(
    divisões=("rodada", "size"), primeira=("rodada", "min"), α_primeira=("α_cross^max", "first"),
    voltas=("voltou na rodada", lambda x: int(x.notna().sum()))).reset_index()